# ThreadCraft — Dataset prep & cleaning: garment category classifier

**Source dataset:** [`ashraq/fashion-product-images-small`](https://huggingface.co/datasets/ashraq/fashion-product-images-small) on the Hugging Face Hub.

Why this one, over the raw Kaggle `paramaggarwal/fashion-product-images-small`:
- Already on HF — `load_dataset(...)` works with **no Kaggle credentials at all**, just this dataset.
- Pre-cleaned to **44,072 rows** (the raw Kaggle CSV has malformed rows that break `pd.read_csv` — already handled here).
- Images are stored as a native HF `Image` feature (decoded PIL images), no manual unzip/path-matching needed.
- MIT-licensed metadata, 60×80 px images — small enough to fine-tune quickly on a Kaggle T4.

**What this notebook does:**
1. Log in to Hugging Face (token from a Kaggle Secret — never hardcoded)
2. Load the raw dataset and inspect it
3. Clean it: drop nulls/dupes, collapse the long tail of rare classes
4. Build a stratified train/val/test split
5. Push the **cleaned** dataset to your own HF Hub repo, so the training notebook loads a ready-to-use `DatasetDict` in one line

Run this as **Kaggle → Save & Run All (commit)** so it executes headlessly and the outputs are preserved even if you close the browser.

## 0. Setup

Before running on Kaggle:
1. **Create an HF token**: https://huggingface.co/settings/tokens → "New token" → role **Write** (you need write access to push the cleaned dataset back).
2. On this Kaggle notebook: **Add-ons → Secrets → Add a new secret** → name it `HF_TOKEN`, paste the token value. Never paste the raw token into a notebook cell — Kaggle notebooks are frequently made public, and a leaked write token lets anyone push to your HF account.
3. Set `HF_USERNAME` below to your Hugging Face username (not your Kaggle username).

In [ ]:
HF_USERNAME = "your-hf-username"  # <-- CHANGE THIS
CLEANED_REPO_ID = f"{HF_USERNAME}/threadcraft-fashion-cleaned"

SOURCE_DATASET = "ashraq/fashion-product-images-small"
TARGET_COLUMN = "subCategory"   # classification target — see "why subCategory" markdown below
MIN_EXAMPLES_PER_CLASS = 40      # classes with fewer rows than this get merged into "Other"
VAL_FRACTION = 0.10
TEST_FRACTION = 0.10
RANDOM_SEED = 42

In [ ]:
!pip install -q -U datasets huggingface_hub scikit-learn

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
login(token=HF_TOKEN)
print("Logged in to Hugging Face.")

## 1. Load and inspect the raw dataset

In [ ]:
from datasets import load_dataset

raw = load_dataset(SOURCE_DATASET, split="train")
print(raw)
print(raw.features)

In [ ]:
import pandas as pd

# Metadata only (drop the image column) for fast EDA — avoids decoding 44k images into memory.
meta = raw.remove_columns("image").to_pandas()
print(f"Rows: {len(meta)}")
meta.head()

In [ ]:
print("Nulls per column:")
print(meta.isnull().sum())
print()
print("Exact duplicate metadata rows:", meta.duplicated().sum())
print()
print(f"'{TARGET_COLUMN}' value counts:")
print(meta[TARGET_COLUMN].value_counts())

### Why `subCategory` and not `articleType`

`articleType` has **141 classes** with a **~7,065:1 imbalance ratio** (Tshirts down to singleton classes like "Cushion Covers") — a realistic target for a research paper, not for a fine-tune you need finished in under an hour with a demo-ready accuracy number. `subCategory` (**45 classes**, top class ~15,383 rows) is still genuinely useful for ThreadCraft (it distinguishes Topwear/Bottomwear/Innerwear/Bags/etc., which maps onto real garment categories) and is tractable in the time budget. `masterCategory` (7 classes) is the safe fallback if `subCategory` still trains poorly after cleaning.

## 2. Clean

1. Drop rows with a null target or null `image`.
2. Drop exact duplicate metadata rows.
3. Merge any class with fewer than `MIN_EXAMPLES_PER_CLASS` rows into an `Other` bucket — this is what makes a stratified split possible at all (`train_test_split` throws if a class has fewer members than the number of splits).

In [ ]:
before = len(raw)

# Row-level null/dupe filter, applied to the full dataset (keeps `image` aligned with `meta`).
keep_mask = meta[TARGET_COLUMN].notnull() & ~meta.duplicated(keep="first")
keep_indices = meta.index[keep_mask].tolist()

cleaned = raw.select(keep_indices)
meta_clean = meta.loc[keep_indices].reset_index(drop=True)

print(f"Rows before cleaning: {before}")
print(f"Rows after null/dupe filter: {len(cleaned)} (-{before - len(cleaned)})")

In [ ]:
counts = meta_clean[TARGET_COLUMN].value_counts()
rare_classes = counts[counts < MIN_EXAMPLES_PER_CLASS].index.tolist()
print(f"Merging {len(rare_classes)} rare classes into 'Other': {rare_classes}")

meta_clean["label"] = meta_clean[TARGET_COLUMN].where(
    ~meta_clean[TARGET_COLUMN].isin(rare_classes), "Other"
)

final_counts = meta_clean["label"].value_counts()
print(f"\nFinal class count: {meta_clean['label'].nunique()}")
print(final_counts)

## 3. Stratified train / validation / test split

In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np

labels = sorted(meta_clean["label"].unique())
label2id = {name: i for i, name in enumerate(labels)}
id2label = {i: name for name, i in label2id.items()}

all_idx = np.arange(len(meta_clean))
stratify_col = meta_clean["label"].values

train_idx, temp_idx = train_test_split(
    all_idx,
    test_size=VAL_FRACTION + TEST_FRACTION,
    stratify=stratify_col,
    random_state=RANDOM_SEED,
)
temp_labels = stratify_col[temp_idx]
val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=TEST_FRACTION / (VAL_FRACTION + TEST_FRACTION),
    stratify=temp_labels,
    random_state=RANDOM_SEED,
)

print(f"Train: {len(train_idx)}  Val: {len(val_idx)}  Test: {len(test_idx)}")

In [ ]:
from datasets import DatasetDict, ClassLabel

class_label = ClassLabel(names=labels)

def attach_label_column(hf_ds, indices):
    subset = cleaned.select(indices.tolist())
    label_ids = [label2id[l] for l in meta_clean["label"].values[indices]]
    subset = subset.add_column("label", label_ids)
    subset = subset.cast_column("label", class_label)
    # Drop the free-text columns the classifier doesn't need — keeps the pushed dataset small.
    drop_cols = [c for c in ["productDisplayName", "year"] if c in subset.column_names]
    return subset.remove_columns(drop_cols)

dataset_dict = DatasetDict({
    "train": attach_label_column(cleaned, train_idx),
    "validation": attach_label_column(cleaned, val_idx),
    "test": attach_label_column(cleaned, test_idx),
})

dataset_dict

## 4. Sanity-check a few images before pushing

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 6, figsize=(15, 3))
sample = dataset_dict["train"].shuffle(seed=RANDOM_SEED).select(range(6))
for ax, row in zip(axes, sample):
    ax.imshow(row["image"])
    ax.set_title(id2label[row["label"]], fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 5. Push the cleaned dataset + label map to your own HF Hub repo

This is what makes the *training* notebook trivial: it just does `load_dataset(CLEANED_REPO_ID)` and gets train/validation/test splits with an already-correct `ClassLabel` column, no re-cleaning.

In [ ]:
dataset_dict.push_to_hub(CLEANED_REPO_ID, private=False)
print(f"Pushed to https://huggingface.co/datasets/{CLEANED_REPO_ID}")

In [ ]:
import json
from huggingface_hub import HfApi, hf_hub_download

with open("label2id.json", "w") as f:
    json.dump(label2id, f, indent=2)

api = HfApi(token=HF_TOKEN)
api.upload_file(
    path_or_fileobj="label2id.json",
    path_in_repo="label2id.json",
    repo_id=CLEANED_REPO_ID,
    repo_type="dataset",
)
print("label2id.json uploaded alongside the dataset.")
print(label2id)

## Next step

In a new Kaggle notebook (GPU T4 x2 enabled), fine-tune `google/vit-base-patch16-224-in21k` on `f"{HF_USERNAME}/threadcraft-fashion-cleaned"`:

```python
from datasets import load_dataset
ds = load_dataset(f"{HF_USERNAME}/threadcraft-fashion-cleaned")
# ds["train"], ds["validation"], ds["test"] are ready to use
```

See the setup guide for the Kaggle GPU + secrets steps — the same `HF_TOKEN` secret works for both notebooks.